[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/agent-observability-course/16_slos_alertas_incidentes/16_slos_alertas_incidentes.ipynb)

# Módulo 16 · SLOs, alertas e incidentes: de «me parece que va mal» a «el presupuesto se agota en 2 horas»

| | |
|---|---|
| **Nivel** | 🔴 Experto |
| **Duración estimada** | ≈ 4 h 30 min (lección ≈ 2 h 45 min + proyecto ≈ 1 h 45 min) |
| **Servicios de Compose** | Opcional: `--profile core --profile metrics` (sección 7.1, ≈ 3 min en vivo). Sin Docker todo funciona en **modo degradado** (simulaciones del agente en memoria). Funciona en Colab |
| **RAM aproximada** | < 500 MB sin stack · ≈ 1,2 GB con `core` + `metrics` |
| **Prerrequisitos** | Módulos 09 (métricas), 10 (alertas básicas y dashboards), 13 (coste), 14 (calidad online) y 15 (regresión) |
| **Hilo conductor** | SupportOps ya mide latencia, coste y calidad. Hoy les ponemos **objetivos**, **alertas que despiertan solo cuando importa**, un **runbook** y un **postmortem** |

> ⚠️ **Honestidad sobre el laboratorio.** El tráfico y los incidentes son **simulados y deterministas** (`mock`): el agente real de SupportOps procesa 24 h de tickets con un fallo masivo
> del proveedor LLM y una regresión de prompt inyectados, de modo que conocemos la verdad de cada alerta. Las ventanas del laboratorio en vivo están **comprimidas** (minutos en vez de horas); las
> ventanas de producción (1 h/5 min, 6 h/30 min…) las tratamos con simulaciones y con aritmética.

## Objetivos de aprendizaje

Al terminar esta lección podrás:

1. **Definir** SLIs de un agente (disponibilidad, latencia, calidad, coste) como «eventos buenos / eventos válidos» y fijar un SLO realista con la línea base medida.
2. **Calcular** presupuesto de error, *burn rate* y tiempo hasta agotarlo; derivar el 14,4 de las políticas de la SRE Workbook.
3. **Implementar** alertas por *burn rate* multiventana (ventana larga **y** corta) y compararlas con un umbral fijo y con una sola ventana en detección, falsos positivos y tiempo de reajuste.
4. **Dimensionar** ventanas para SLIs de poco volumen (calidad evaluada por muestreo) y entender por qué una regresión de calidad es una quema **lenta**.
5. **Escribir** reglas de alerta de Grafana **provisionadas como código** (intervalo múltiplo de 10 s, vínculo alerta→panel en `annotations`) y probarlas con un incidente real en el stack.
6. **Diseñar** una política de presupuesto de error, un runbook con árbol de diagnóstico y un esquema de guardia que no genere fatiga.
7. **Redactar** un postmortem **sin culpa** con cifras sacadas de la telemetría (TTD, impacto, presupuesto consumido) y acciones comprobables.

## 🔁 Conexión con módulos anteriores

<details>
<summary>1 · (Módulo 10) ¿Por qué el intervalo de un grupo de alertas de Grafana debe ser múltiplo de 10 s y cómo se enlaza una alerta con un panel?</summary>

Porque el planificador de Grafana evalúa en ticks de 10 s: un intervalo como `15s` hace que Grafana no arranque. El enlace va en `annotations.__dashboardUid__` y `annotations.__panelId__` (el on-call llega a la gráfica que importa).

</details>

<details>
<summary>2 · (Módulo 14) ¿Por qué la calidad se mide con una muestra y qué implica para un SLI de calidad?</summary>

Porque evaluar cuesta (≈ 9× el coste de atender). Implica **poco volumen** por ventana: el SLI de calidad es ruidoso y necesita ventanas más largas y un mínimo de eventos.

</details>

<details>
<summary>3 · (Módulo 15) ¿Qué señal delató la regresión silenciosa y por qué ninguna alerta de latencia/errores la vio?</summary>

La caída de groundedness por versión y la huella de comportamiento (menos herramientas por ticket). Latencia y errores no la ven porque la petición termina bien y más rápido.

</details>

In [ ]:
# Instala el paquete del curso (editable) con extras de gráficos. Sirve en local y en Colab (clona el repo si hace falta).
import pathlib, subprocess, sys

def _find_root():
    for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (p / "pyproject.toml").exists() and (p / "supportops").is_dir():
            return p
    return None

ROOT = _find_root()
if ROOT is None:  # p. ej. Google Colab: el notebook se abre sin el repo
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/padillacm/IA-practice", "/content/IA-practice"], check=True)
    ROOT = pathlib.Path("/content/IA-practice/agent-observability-course")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[viz,otel]"], check=True)
if str(ROOT) not in sys.path:   # un editable recién instalado no es visible hasta reiniciar el kernel: lo añadimos a mano
    sys.path.insert(0, str(ROOT))
import importlib; importlib.invalidate_caches()
print("Curso en:", ROOT)

In [ ]:
import os
os.environ.setdefault("LLM_PROVIDER", "mock")          # mock = determinista, sin coste, sin API keys
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from supportops import *                                 # SupportAgent, generate_tickets, check_stack, ...
from supportops import viz, sample_data, schema, backends
viz.apply_style()
pd.set_option("display.width", 140); pd.set_option("display.max_colwidth", 80)
SEED = 42
FAST_DEV_RUN = True   # True: pocos tickets (iteración rápida). False: tamaños «completos» del texto

import collections, dataclasses, json, math, re, time, threading
import yaml
from IPython.display import Markdown
from supportops.slo import (BASELINE_FAULTS, DEFAULT_SLOS, GOOGLE_POLICIES, LAB_POLICIES, DayScenario, MultiWindowPolicy, alert_quality, bin_events, budget_report,
                            burn_promql, burn_series, fire_series, grafana_burn_rule, minute_mask, pages_per_week, render_postmortem, rules_yaml, simulate_day,
                            simulate_month, sli_promql, threshold_fire_series, trailing_sum)
STACK = check_stack("core+metrics")
LIVE = not STACK.degraded

In [ ]:
DAY = simulate_day(SEED)                                   # 24 h de tráfico REAL del agente (tiempo simulado): incidente de proveedor 10:00-10:45 + regresión de prompt v2 06:00-14:00
ROWS, NB, T0, TRUTH = DAY["rows"], DAY["n_bins"], DAY["t0"], DAY["truth"]
OUT, REG = minute_mask(*TRUTH["outage"], NB), minute_mask(*TRUTH["regression"], NB)
SERIES = {name: bin_events(ROWS, slo, t0=T0, n_bins=NB) for name, slo in DEFAULT_SLOS.items()}       # (buenos, totales) por minuto
print(f"{len(ROWS)} tickets en 24 h · incidente del proveedor: min {TRUTH['outage'][0]:.0f}-{TRUTH['outage'][1]:.0f} · regresión de prompt: min {TRUTH['regression'][0]:.0f}-{TRUTH['regression'][1]:.0f}")
print({n: f"{g.sum() / t.sum():.4f} (objetivo {DEFAULT_SLOS[n].objective:.2%}, {int(t.sum())} eventos)" for n, (g, t) in SERIES.items()})

---
# 1 · Intuición: el contrato de fiabilidad

Un **SLO** (*Service Level Objective*) es una promesa interna, medible y revisable: «el 99 % de los tickets termina sin error cada mes». Su gracia no es el número sino lo que habilita:
**un presupuesto de error** (el 1 % restante) que el equipo puede *gastar* en desplegar, experimentar y fallar, y una **alerta** que salta cuando ese presupuesto se consume a un ritmo peligroso — ni antes (ruido) ni después (clientes quemados).

| Si vienes de… | Esto es como… | Lo que cambia |
|---|---|---|
| **ML** | Un umbral de métrica en producción (p. ej. AUC mínimo) | El umbral es una **tasa de eventos buenos** sobre una ventana y lleva un presupuesto asociado |
| **Backend** | SLA/SLO de una API (p. ej. 99,9 % de peticiones < 300 ms) | Un agente necesita SLOs de **calidad** y **coste**, no solo de disponibilidad y latencia |
| **SRE** | Google SRE: SLI → SLO → error budget → alertas por burn rate → postmortem | Es exactamente el mismo ciclo; lo nuevo son los SLIs |

In [ ]:
fig, ax = viz.canvas(12.5, 4.4, (0, 125), (0, 44))
P = viz.PALETTE
bs = [viz.box(ax, 1 + i * 20.5, 24, 18, 9, t, c, fontsize=8) for i, (t, c) in enumerate([("SLI\neventos buenos / válidos", P["metrics"]), ("SLO\nobjetivo 99 % / 30 d", P["traces"]), ("Presupuesto de error\n(1 − objetivo)", P["warn"]),
                                                                                    ("Burn rate\n= ritmo de gasto", P["metrics"]), ("Alerta multiventana\npágina / ticket", P["error"]), ("Runbook +\nguardia", P["logs"])])]
for a, b in zip(bs, bs[1:]): viz.arrow(ax, viz.right(a), viz.left(b))
b_pm = viz.box(ax, 41, 5, 40, 9, "Postmortem sin culpa → acciones → revisar SLO, alertas y runbook", P["profiles"], fontsize=9)
viz.arrow(ax, viz.bottom(bs[4]), viz.top(b_pm), "", P["profiles"]); viz.arrow(ax, viz.left(b_pm), (bs[0][0] + 9, 22.5), "", P["profiles"], rad=-0.35, dashed=True)
b_pol = viz.box(ax, 90, 5, 33, 9, "Política: presupuesto agotado\n→ congelar despliegues", P["neutral"], fc="#f3f4f6", fontsize=8)
viz.arrow(ax, viz.top(bs[2]), (106, 14.3), "", P["neutral"], rad=-0.3, dashed=True)
ax.text(1, 40, "El ciclo SLO: medir → prometer → gastar → alertar → aprender", fontsize=10, fontweight="bold")
plt.show()

> 👀 **Qué debes observar**
>
> - El ciclo es **cerrado**: el postmortem retroalimenta los SLI, el objetivo, las alertas y el runbook. Un SLO que nunca se revisa se convierte en decoración.
> - El presupuesto de error es una **herramienta de negociación** entre producto y fiabilidad: con presupuesto, se despliega; sin presupuesto, se congela y se arregla. Hace objetiva una discusión que suele ser política.
> - En un agente, lo que consume presupuesto no es solo «caída del servicio»: puede ser una **regresión de calidad** o una **explosión de coste**; por eso hay un SLO por dimensión.

---
# 2 · Teoría: SLI, SLO, presupuesto y burn rate

**SLI** (indicador): \( \text{SLI} = \dfrac{\text{eventos buenos}}{\text{eventos válidos}} \) en una ventana. Dos decisiones de diseño: qué es un evento **válido** (¿cuentan los tickets de prueba?) y cuál es el **bueno**.

**SLO** (objetivo): \( \text{SLI} \ge \text{objetivo} \) en un periodo \(T\) (típico: 28–30 días, móvil).

**Presupuesto de error:** \( B = 1-\text{objetivo} \): la fracción de eventos válidos que pueden ser malos en \(T\). Con objetivo 99 % y 1 000 000 de tickets al mes, \(B\) = 10 000 tickets malos.

**Burn rate** (ritmo de consumo) en una ventana \(w\): \( \text{burn}(w) = \dfrac{1-\text{SLI}(w)}{B} \). Un burn de 1 consume exactamente el presupuesto en \(T\); de 14,4 lo agota en \(T/14{,}4\).

**Presupuesto consumido** manteniendo un burn \(r\) durante \(w\): \( \dfrac{r\cdot w}{T} \). El 14,4 de la quema rápida sale de fijar **«el 2 % del presupuesto de un mes en 1 hora»**: \( r = 0{,}02\cdot 720\,\text{h}/1\,\text{h} = 14{,}4 \).

In [ ]:
rows = []
for pol in GOOGLE_POLICIES:
    rows.append({"política": pol.name, "burn mínimo": pol.factor, "ventana larga": f"{pol.long_min // 60} h" if pol.long_min < 1440 else f"{pol.long_min // 1440} d", "ventana corta": f"{pol.short_min} min" if pol.short_min < 60 else f"{pol.short_min // 60} h",
                 "presupuesto consumido": f"{pol.budget_consumed():.0%}", "severidad": pol.severity, "tiempo hasta agotar todo el presupuesto": f"{30 * 24 / pol.factor:.0f} h"})
display(pd.DataFrame(rows).set_index("política"))
print("Comprobación: 14,4 × 1 h / 720 h =", 14.4 * 1 / 720, "· 6 × 6 h / 720 h =", 6 * 6 / 720, "· 1 × 72 h / 720 h =", 72 / 720)

> 👀 **Qué debes observar**
>
> - Las tres políticas de la SRE Workbook para un SLO a 30 días son: **14,4 / 1 h** (2 % del presupuesto, *página*), **6 / 6 h** (5 %, *página*) y **1 / 3 d** (10 %, *ticket*). Cada ventana corta es ≈ 1/12 de la larga.
> - Un burn alto con ventana corta detecta rápido lo grave; un burn bajo con ventana larga detecta lo lento que, sumado, también agota el presupuesto. **Hacen falta ambas**: una sola nunca cubre los dos extremos.
> - Con un objetivo de 99 %, \(B\) = 1 %: un burn de 14,4 equivale a un 14,4 % de tickets malos. Con un objetivo de 99,9 %, el mismo burn es un 1,44 %: el umbral absoluto depende del objetivo, el burn no.

> **⚠️ Cuidado · Cuidado: fórmulas y ventanas del laboratorio**
>
> En el laboratorio en vivo (sección 7.1) las ventanas están **comprimidas** (5 min/1 min para la quema rápida, 15 min/3 min para la lenta) porque no podemos esperar una hora. Los **factores** (14,4 y 3) se mantienen:
> la aritmética «consumo = burn × ventana / periodo» sigue valiendo si interpretas el periodo como 30 min. En producción usa las ventanas de la tabla.

---
# 3 · SLIs específicos de un agente

| SLI | Evento válido | Evento bueno | Dónde se mide | Por qué es de *agente* |
|---|---|---|---|---|
| **Disponibilidad** | Ticket procesado | `status` ∈ {`ok`, `escalated`} (no error, degradado ni `max_steps`) | `supportops_agent_requests_total{supportops_status}` | Un «degradado» responde «problema técnico»: para el cliente es una caída |
| **Latencia** | Ticket procesado | Duración de `invoke_agent` ≤ 5 s | `gen_ai_invoke_agent_duration_seconds_bucket{le="5.0"}` | Percentil de **extremo a extremo**, incluidas herramientas y reintentos |
| **Calidad** | Ticket **evaluado** (muestra uniforme del 25 %) | `groundedness` ≥ 0,5 según el juez | `supportops_eval_results_total` (módulo 14) | Una respuesta rápida e inventada es un fallo |
| **Coste** | Ticket procesado | Coste ≤ 0,0005 USD (precios ilustrativos) | `supportops_agent_request_cost_USD_bucket` (módulo 13) | Un bucle de herramientas es un incidente aunque «funcione» |

Dos reglas para elegir SLIs: (1) que midan lo que **percibe el usuario**, no una causa interna (CPU del Collector no es un SLI); (2) que sean **expresables como ratio bueno/válido**, para que el burn rate se calcule igual en todos.
La calidad usa una **muestra uniforme** (no la estratificada del módulo 14) porque el SLI debe ser simple y sin sesgo: la muestra «de caza» (errores, valoraciones negativas) es para investigar, la «de medir» es para el SLO.

In [ ]:
def rolling(g, t, w=30):
    gg, tt = trailing_sum(g, w), trailing_sum(t, w)
    with np.errstate(divide="ignore", invalid="ignore"): return np.where(tt >= 5, gg / tt, np.nan)
fig, axes = plt.subplots(2, 2, figsize=(13, 6.4), sharex=True)
hours = np.arange(NB) / 60
for ax, (name, slo) in zip(axes.ravel(), DEFAULT_SLOS.items()):
    g, t = SERIES[name]
    ax.axvspan(*np.array(TRUTH["regression"]) / 60, color=viz.PALETTE["warn"], alpha=0.12); ax.axvspan(*np.array(TRUTH["outage"]) / 60, color=viz.PALETTE["error"], alpha=0.2)
    ax.plot(hours, rolling(g, t), color=viz.PALETTE["traces"], lw=1.3); ax.axhline(slo.objective, color=viz.PALETTE["ok"], ls="--", label=f"objetivo {slo.objective:.0%}")
    ax.set_title(f"SLI {name} (media móvil 30 min)"); ax.set_ylim(max(0.3, min(np.nanmin(rolling(g, t)) - 0.02, slo.objective - 0.05)), 1.005); ax.legend(loc="lower left", fontsize=8)
for ax in axes[1]: ax.set_xlabel("hora del día (simulada)")
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar**
>
> - La franja roja (incidente del proveedor 10:00-10:45) hunde la **disponibilidad** muy por debajo del objetivo y, curiosamente, deja la **latencia** igual o mejor (los errores son rápidos). Mirar solo una dimensión da una historia incompleta.
> - La franja amarilla (regresión de prompt, 6:00-14:00) **no toca** disponibilidad ni latencia: solo se ve en el SLI de **calidad**, que cae durante 8 horas por debajo del objetivo. Es el patrón del módulo 15 convertido en SLO.
> - El **coste** apenas se mueve en este día: un SLO que casi nunca se acerca al umbral protege contra algo que *no ha ocurrido hoy* (bucles, cambio de modelo, módulo 13). Tener un SLI «aburrido» es lo normal y deseable.

In [ ]:
rows = []
for name, slo in DEFAULT_SLOS.items():
    g, t = SERIES[name]
    month_events = t.sum() * 30                         # el presupuesto del MES: si todos los días fueran como hoy, ¿qué fracción del presupuesto mensual se gasta hoy?
    rep = budget_report(g.sum(), t.sum(), slo.objective)
    rows.append({"SLO": name, "objetivo": f"{slo.objective:.1%}", "SLI del día": g.sum() / t.sum(), "eventos malos": int(rep["bad"]), "presupuesto mensual (eventos malos)": int(slo.budget * month_events),
                 "% del presupuesto MENSUAL gastado hoy": 100 * rep["bad"] / (slo.budget * month_events), "burn medio del día": rep["avg_burn_rate"]})
BR = pd.DataFrame(rows).set_index("SLO"); display(BR.round(3))

> 👀 **Qué debes observar**
>
> - Un solo día malo puede gastar buena parte del presupuesto **mensual**: la disponibilidad se lleva ≈ 12 % del mes en un día con 45 minutos de incidente. Con ocho días así el SLO estaría incumplido.
> - Fíjate en la columna «burn medio del día»: la calidad quema por encima de 1 **todo el día** aunque el valor absoluto parezca pequeño; la disponibilidad lo hace de golpe. Son dos *formas* de gastar presupuesto que piden alertas distintas (secciones 4-5).

---
# 4 · Alertar por burn rate (a mano) y por qué una ventana no basta

Implementamos el cálculo (sin librería): suma móvil de buenos y totales en la ventana, ratio de malos y división por el presupuesto. Después comparamos **cuatro estrategias** sobre el mismo día:

| Estrategia | Regla |
|---|---|
| **Umbral fijo** (anti-patrón) | tasa de error en 5 min > 5 % |
| **Burn corto solo** | burn(5 min) ≥ 14,4 |
| **Burn largo solo** | burn(1 h) ≥ 14,4 |
| **Multiventana** | burn(1 h) ≥ 14,4 **y** burn(5 min) ≥ 14,4 |

In [ ]:
def burn_by_hand(good, total, width, objective, min_events=5):
    g = np.convolve(good, np.ones(width), "full")[: len(good)]; t = np.convolve(total, np.ones(width), "full")[: len(total)]      # suma móvil (ventana de `width` minutos)
    bad_ratio = np.where(t >= min_events, 1 - g / np.maximum(t, 1), np.nan)
    return bad_ratio / (1 - objective)
g, t = SERIES["availability"]; slo = DEFAULT_SLOS["availability"]
assert np.allclose(np.nan_to_num(burn_by_hand(g, t, 60, slo.objective)), np.nan_to_num(burn_series(g, t, 60, slo.objective)))          # a mano = librería
b5, b60 = burn_by_hand(g, t, 5, slo.objective), burn_by_hand(g, t, 60, slo.objective)
strat = {"Umbral fijo (5 min > 5 %)": threshold_fire_series(g, t, 0.05, 5), "Burn corto solo (5 min)": np.nan_to_num(b5) >= 14.4, "Burn largo solo (1 h)": np.nan_to_num(b60) >= 14.4,
         "Multiventana (1 h y 5 min)": fire_series(g, t, slo.objective, GOOGLE_POLICIES[0])}
inc_end = int(TRUTH["outage"][1]); res = []
for k, f in strat.items():
    q = alert_quality(f, OUT, reset_slack_min=0)
    late = int(f[inc_end:].sum()); outside = int((f & ~OUT).sum())
    res.append({"estrategia": k, "detecta": q["detected"], "TTD (min)": q["ttd_min"], "minutos disparada": q["fired_intervals"], "fuera del incidente (min)": outside, "tras acabar el incidente (min)": late})
display(pd.DataFrame(res).set_index("estrategia"))
fig, ax = plt.subplots(figsize=(11, 3.2))
for i, (k, f) in enumerate(strat.items()):
    ax.fill_between(hours, i, i + 0.8, where=f, color=viz.PALETTE["error"], step="mid"); ax.text(8.55, i + 0.35, k, fontsize=8, va="center")
ax.axvspan(*np.array(TRUTH["outage"]) / 60, color=viz.PALETTE["error"], alpha=0.15); ax.set_yticks([]); ax.set_xlabel("hora"); ax.set_xlim(8.5, 13); ax.set_title("Cuándo dispara cada estrategia (franja = incidente real)")
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar**
>
> - **Umbral fijo**: detecta al instante, pero con 2-4 tickets por minuto basta un par de fallos seguidos para disparar: ≈ 70 minutos de alerta **fuera** del incidente en un solo día. Es el generador de ruido que acaba con la guardia.
> - **Burn largo solo (1 h)**: detecta a los ≈ 8 min (necesita acumular un 14,4 % de malos en la ventana) y **sigue disparada ≈ 45 min después de que el incidente termine** (la ventana larga recuerda): el on-call recibe una alerta de algo que ya está arreglado.
> - **Burn corto solo (5 min)**: detecta y se reajusta rápido, pero sigue disparando por ráfagas ajenas al incidente (≈ 10 min fuera): sin ventana larga, un blip de 5 minutos dispara igual.
> - **Multiventana**: detecta igual de pronto que la larga (≈ 8 min) y la corta **apaga la alerta en cuanto el problema cesa** (≈ 4 min tras el fin frente a ≈ 45): es la razón de ser del diseño de la SRE Workbook.

## 4.1 Monte Carlo: un mes con ruido

El día de ejemplo es un solo caso. Simulamos **40 meses** (binomial por minuto; el agente completo para 40 × 43 200 min sería demasiado) con 3 incidentes reales por mes de gravedad aleatoria y una **ráfaga corta cada día** (3 min al 50 % de fallos) que un buen sistema de alertas no debe convertir en página.

In [ ]:
def month(seed):
    rng = np.random.default_rng(1000 + seed)
    inc = [(int(rng.integers(1, 29)), float(rng.uniform(0, 22)), float(rng.uniform(20, 120)), float(rng.uniform(0.12, 0.7))) for _ in range(3)]
    blips = [(d, float(rng.uniform(0, 23.5)), 3.0, 0.5) for d in range(30)]
    good, total = simulate_month(seed, incidents=inc + blips)
    real = np.zeros(len(good), bool)
    for d, h, dur, r in inc: real[int(d * 1440 + h * 60): int(d * 1440 + h * 60 + dur) + 60] = True               # incidente + 60 min de holgura de reajuste
    return good, total, real, inc
def episodes(mask): idx = np.where(mask)[0]; return [] if not len(idx) else np.split(idx, np.where(np.diff(idx) > 30)[0] + 1)
STRATS = {"umbral fijo (5 min > 5 %)": lambda g, t: threshold_fire_series(g, t, 0.05, 5), "burn 5 min solo": lambda g, t: np.nan_to_num(burn_series(g, t, 5, 0.99)) >= 14.4,
          "burn 1 h solo": lambda g, t: np.nan_to_num(burn_series(g, t, 60, 0.99)) >= 14.4, "multiventana (1 h y 5 min)": lambda g, t: fire_series(g, t, 0.99, GOOGLE_POLICIES[0])}
agg = {k: {"incidentes detectados": 0, "incidentes": 0, "páginas": 0, "páginas falsas": 0, "tarde": 0} for k in STRATS}
for sd in range(40):
    g, t, real, inc = month(sd)
    for k, fn in STRATS.items():
        f = fn(g, t); eps = episodes(f)
        agg[k]["páginas"] += len(eps); agg[k]["páginas falsas"] += sum(not real[e].any() for e in eps)
        for d, h, dur, r in inc:
            agg[k]["incidentes"] += 1; s0 = int(d * 1440 + h * 60); agg[k]["incidentes detectados"] += bool(f[s0: s0 + int(dur) + 5].any()); agg[k]["tarde"] += int(f[s0 + int(dur): s0 + int(dur) + 60].sum())
M = pd.DataFrame(agg).T
M["detección"] = M["incidentes detectados"] / M["incidentes"]; M["páginas/semana"] = M["páginas"] / (40 * 30 / 7); M["% de páginas falsas"] = M["páginas falsas"] / M["páginas"]
M["min. de alerta tras acabar (por incidente)"] = M["tarde"] / M["incidentes"]
display(M[["detección", "páginas/semana", "% de páginas falsas", "min. de alerta tras acabar (por incidente)"]].round(3))
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
axes[0].bar(range(4), M["detección"], color=viz.PALETTE["traces"]); axes[0].set_xticks(range(4), [k.replace(" (", "\n(") for k in M.index], fontsize=7); axes[0].set_ylim(0, 1.05); axes[0].set_title("Incidentes reales detectados")
axes[1].bar(range(4), M["páginas/semana"] * (1 - M["% de páginas falsas"]), color=viz.PALETTE["ok"], label="páginas reales"); axes[1].bar(range(4), M["páginas/semana"] * M["% de páginas falsas"], bottom=M["páginas/semana"] * (1 - M["% de páginas falsas"]), color=viz.PALETTE["error"], label="páginas FALSAS")
axes[1].set_xticks(range(4), [k.replace(" (", "\n(") for k in M.index], fontsize=7); axes[1].legend(); axes[1].set_title("Páginas por semana de guardia")
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar**
>
> - El **umbral fijo** detecta todo, pero **≈ 99 % de sus páginas son falsas** (≈ 66 por semana, con las ráfagas diarias): el camino directo a la fatiga de alertas y a ignorar la siguiente página, que sí era real. El **burn de 5 min solo** mejora pero aún da ≈ 11 páginas/semana, ≈ 93 % falsas.
> - Las dos políticas con ventana larga (1 h solo y multiventana) **eliminan las páginas falsas** (≈ 0,6 páginas/semana, todas reales) y detectan los incidentes que gastan presupuesto de verdad; el ≈ 8 % restante son incidentes tan suaves (≈ 12-14 % de fallos) que el burn no llega a 14,4 y no hace falta despertar a nadie. Detectar un 100 % no es el objetivo: es detectar lo que **agota** el presupuesto.
> - ¿Dónde está entonces la ventaja de la multiventana sobre la ventana larga sola? En la última columna: **sigue disparada muchos menos minutos después de acabar el incidente** (la corta la apaga). El recuento de páginas no ve ese coste; la guardia sí (alertas «fantasma» de algo ya resuelto).
> - Mide siempre tu política con **páginas por semana, % de acciones útiles y tiempo de reajuste** — la métrica de salud de la guardia.

---
# 5 · La calidad: una quema lenta con poco volumen

La regresión de prompt (6:00-14:00) consume presupuesto de calidad **sostenidamente** por encima de 1 pero muy por debajo de 14,4. Dos problemas: (a) ¿qué política la ve? y (b) el SLI de calidad se mide en una **muestra** (25 %): pocos eventos por ventana.

In [ ]:
gq, tq = SERIES["quality"]; slq = DEFAULT_SLOS["quality"]
vol = [(w, tq[300:300 + w].sum()) for w in (5, 15, 30, 60, 180, 360)]
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
axes[0].plot([w for w, _ in vol], [v for _, v in vol], "o-", color=viz.PALETTE["metrics"]); axes[0].axhline(30, color=viz.PALETTE["error"], ls=":"); axes[0].text(6, 33, "≈ 30 eventos: mínimo razonable", fontsize=8)
axes[0].set_xscale("log"); axes[0].set_xlabel("ventana (min)"); axes[0].set_ylabel("evaluaciones en la ventana"); axes[0].set_title("Volumen del SLI de calidad por ventana")
for w, col_ in ((30, viz.PALETTE["traces"]), (180, viz.PALETTE["profiles"]), (360, viz.PALETTE["metrics"])):
    b = burn_series(gq, tq, w, slq.objective, min_events=30); axes[1].plot(hours, b, label=f"ventana {w} min", color=col_)
axes[1].axhline(1, color=viz.PALETTE["ok"], ls="--", label="ritmo sostenible (1×)"); axes[1].axhline(6, color=viz.PALETTE["error"], ls=":", label="6×"); axes[1].axvspan(*np.array(TRUTH["regression"]) / 60, color=viz.PALETTE["warn"], alpha=0.12)
axes[1].set_xlabel("hora"); axes[1].set_ylabel("burn rate de calidad"); axes[1].set_ylim(0, 12); axes[1].legend(fontsize=7); axes[1].set_title("Burn rate de calidad con distintas ventanas")
plt.tight_layout(); plt.show()
reg = slice(int(TRUTH["regression"][0]) + 60, int(TRUTH["regression"][1]))
print(f"durante la regresión: SLI de calidad {gq[reg].sum() / tq[reg].sum():.3f} (objetivo {slq.objective}) -> burn medio {(1 - gq[reg].sum() / tq[reg].sum()) / slq.budget:.1f}× · fuera: {gq[:300].sum() / tq[:300].sum():.3f}")
for pol in GOOGLE_POLICIES:
    f = fire_series(gq, tq, slq.objective, pol); print(f"  {pol.name:12s} (×{pol.factor:g}, {pol.long_min} min / {pol.short_min} min): dispara {int(f.sum())} min" + (f" · primera vez en el minuto {int(np.argmax(f))}" if f.any() else ""))

> 👀 **Qué debes observar**
>
> - Con el 25 % de muestreo y ≈ 2 tickets/min, una ventana de 5 min tiene ≈ 2-3 evaluaciones y una de 30 min, ≈ 15: **el burn de calidad con ventanas cortas es puro ruido** (por eso `min_events`). Para tener ≈ 30 eventos hacen falta ventanas de ≈ 1 h.
> - Durante la regresión el burn de calidad está en ≈ 4×: **muy por encima de 1 pero por debajo de 6 y de 14,4**. Las políticas de página (14,4×/1 h y 6×/6 h) **no disparan nunca**; solo la de ticket (1×) lo detecta, ≈ 45 min después del despliegue (en este día de 24 h la ventana de «3 días» equivale a todo el historial). Es una alerta de **ticket**. Está bien: la regresión consume ≈ 4-5 % del presupuesto mensual en 8 horas y el remedio no es despertar a nadie sino **bloquear el despliegue en canario** (módulo 15) y revisar por la mañana.
> - Moraleja de diseño: la severidad la decide **cuánto presupuesto quema y con qué urgencia puedes actuar**, no lo «grave» que suene la métrica.

> **🧪 Experimento · Experimento: SLI sobre una métrica ruidosa**
>
> Antes de poner un SLO sobre la calidad del juez, mide la tasa de calidad **en una semana sana**: si el juez tiene un 3 % de falsos positivos (módulo 14) y tu objetivo es 98 %, el SLO se incumple solo por el juez.
> Fija el objetivo **por debajo de la línea base medida** (aquí 95 %, con línea base ≈ 96,7 %) y calibra el juez (Rogan–Gladen) si la precisión importa.

---
# 6 · Política de presupuesto de error: un mes completo

El presupuesto se *gasta*: incidentes, despliegues fallidos, mantenimiento. Una **política** acordada de antemano decide qué pasa cuando se agota (típico: congelar despliegues de funcionalidades y dedicar el equipo a fiabilidad). Simulamos un mes con cuatro incidentes.

In [ ]:
incs = [(4, 10, 45, 0.5), (13, 2, 90, 0.2), (21, 15, 30, 0.7), (26, 9, 25, 0.35)]
g, t = simulate_month(SEED, incidents=incs)
slo_a = DEFAULT_SLOS["availability"]
bad_cum = np.cumsum(t - g); allowed_total = slo_a.budget * t.sum()
remaining = 1 - bad_cum / allowed_total
days = np.arange(len(g)) / 1440
fig, ax = plt.subplots(figsize=(10, 3.8))
ax.plot(days, remaining * 100, color=viz.PALETTE["traces"], lw=2); ax.axhline(0, color=viz.PALETTE["error"], ls="--"); ax.fill_between(days, remaining * 100, 0, where=remaining < 0, color=viz.PALETTE["error"], alpha=0.2)
for d, h, dur, r in incs: ax.axvline(d + h / 24, color=viz.PALETTE["warn"], ls=":"); ax.text(d + h / 24 + 0.1, 92, f"{dur} min\n{r:.0%}", fontsize=7)
exhaust = np.argmax(remaining < 0) / 1440 if (remaining < 0).any() else None
if exhaust: ax.axvline(exhaust, color=viz.PALETTE["error"]); ax.text(exhaust + 0.2, -20, f"presupuesto agotado\n día {exhaust:.1f} → congelar despliegues", fontsize=8, color=viz.PALETTE["error"])
ax.set_xlabel("día del mes"); ax.set_ylabel("presupuesto restante (%)"); ax.set_title("Presupuesto de disponibilidad restante a lo largo del mes")
plt.tight_layout(); plt.show()
rep = budget_report(g.sum(), t.sum(), slo_a.objective)
print(f"SLI del mes {rep['sli']:.4f} (objetivo {slo_a.objective:.2%}) · consumido {rep['consumed']:.0%} del presupuesto · restante {rep['remaining']:.0%}")
for i, (d, h, dur, r) in enumerate(incs):
    s0 = int(d * 1440 + h * 60); bad = (t[s0:s0 + dur] - g[s0:s0 + dur]).sum(); print(f"  incidente {i + 1} (día {d}, {dur} min al {r:.0%}): {bad:.0f} tickets malos = {bad / allowed_total:.1%} del presupuesto del mes")

> 👀 **Qué debes observar**
>
> - Cada incidente es un **escalón** en la curva (entre el 2 % y el 7 % del presupuesto cada uno), pero la pendiente entre incidentes —el consumo de fondo, los fallos «normales»— se lleva **más que los cuatro incidentes juntos** (≈ 41 puntos frente a ≈ 18). Con un fondo del 0,4 % de fallos y un objetivo del 99 %, el 40 % del presupuesto ya está comprometido antes del primer incidente: **mide el fondo** antes de prometer un objetivo.
> - Aquí el mes termina con ≈ 40 % de presupuesto: no se activa la congelación. Con un incidente más grande, o un fondo algo peor, la curva cruzaría el 0 y la política diría «congelar» (la celda dibuja la marca si ocurre).
> - Una política útil es **explícita y escalonada**: <50 % restante → revisar despliegues arriesgados; <25 % → solo cambios de fiabilidad; agotado → congelación y postmortem obligatorio. Se acuerda con producto **antes** del primer incidente.

---
# 7 · Alertas de Grafana provisionadas como código

El módulo 10 provisionó alertas de **síntomas** (tasa de error, p95). Hoy provisionamos alertas de **SLO**: 2 políticas × 4 SLOs = 8 reglas, generadas por `supportops.slo.grafana_burn_rule` (el mismo código que prueban las pruebas del paquete) en
`stack/grafana/provisioning/alerting/16-slo-alerts.yaml`. Estructura de cada regla:

| Pieza | Qué hace |
|---|---|
| `A` y `B` | Dos consultas PromQL **instantáneas**: burn rate con la ventana **larga** y con la **corta** |
| `C` y `D` | `reduce` (último valor, descartando NaN) de A y B |
| `E` | `math`: `$C > f && $D > f` — las **dos** ventanas por encima del factor |
| `F` | `threshold` de E (`> 0`) → **condition** de la regla |
| `for` | `0s` en página (la multiventana ya amortigua) · `1m` en ticket |
| `annotations.__dashboardUid__` / `__panelId__` | Enlace alerta → panel del dashboard `slo-agente` |
| `annotations.runbook_url` | El runbook de la alerta |
| `labels.severity` | `page` o `ticket`: la política de notificación (módulo 10) enruta por esta etiqueta |
| `noDataState: OK` | Sin tráfico no hay quema (la alerta de «agente inactivo» es otra) |
| grupo `interval: 30s` | **Múltiplo de 10 s**: el planificador de Grafana evalúa en ticks de 10 s |

In [ ]:
y = yaml.safe_load(open(ROOT / "stack" / "grafana" / "provisioning" / "alerting" / "16-slo-alerts.yaml"))
grp = y["groups"][0]
print("grupo:", grp["name"], "· intervalo:", grp["interval"], "· múltiplo de 10 s:", int(grp["interval"].rstrip("s")) % 10 == 0, "· reglas:", len(grp["rules"]))
display(pd.DataFrame([{"regla": r["title"], "severidad": r["labels"]["severity"], "for": r["for"], "dashboard": r["annotations"]["__dashboardUid__"], "panel": r["annotations"]["__panelId__"]} for r in grp["rules"]]).set_index("regla"))
r0 = grp["rules"][0]
print("\nConsulta A (burn largo) de", r0["title"], ":\n ", r0["data"][0]["model"]["expr"], "\nMath E:", r0["data"][4]["model"]["expression"])
try:
    rules_yaml([r0], interval="15s")
except ValueError as e:
    print("\nintervalo de 15 s ->", e)

> 👀 **Qué debes observar**
>
> - Las 8 reglas apuntan a los paneles `Burn rate · <slo>` (ids 12-15) del dashboard `16-slos.json`: el on-call llega a la gráfica con las **mismas consultas** que disparan la alerta. Mantener alerta y panel sincronizados se hace con código (`burn_promql`), no a mano.
> - `rules_yaml` rechaza un intervalo que no sea múltiplo de 10 s: es un error clásico (Grafana ni arranca con `15s`; módulo 10).
> - Las ventanas del laboratorio (5 m/1 m y 15 m/3 m) son parámetros de `LAB_POLICIES`; para producción se cambia por `GOOGLE_POLICIES` y se regeneran.

## 7.1 Probarlas con un incidente real

Un SLO sin prueba es una creencia. Con el stack arriba, `LivePipeline` envía ≈ 100 s de tráfico a 5 tickets/s con un **fallo masivo del proveedor LLM** entre los 30 s y los 65 s (`Incident("llm_provider_outage", "llm_error", …)`), mientras un hilo consulta el estado de las reglas en Grafana cada 5 s.
Después esperamos a que el burn de la ventana corta baje y la alerta se **resuelva**. Duración total ≈ 3 min. (Si el stack acaba de arrancar, las reglas necesitan unos segundos para cargarse; si no ves nada, `POST /api/admin/provisioning/alerting/reload`.)

In [ ]:
from supportops.regression import LivePipeline
from supportops.quality_eval import RateSampler
from supportops.traffic import Incident
LIVE_S = 100 if FAST_DEV_RUN else 180
OUT_LAB = (30, 65) if FAST_DEV_RUN else (50, 110)
STATES, STOP = [], threading.Event()
def poll_rules(t0):
    while not STOP.is_set():
        try:
            d = backends.grafana_api("/api/prometheus/grafana/api/v1/rules")
            for g_ in d["data"]["groups"]:
                if g_["name"] == "supportops-slo": STATES.append((time.time() - t0, {r["name"]: r["state"] for r in g_["rules"]}))
        except Exception as e: pass
        STOP.wait(5)
if LIVE:
    lp = LivePipeline(("v1",), mode="otlp", sampler=RateSampler(0.3), metric_interval_s=5)
    tk = generate_tickets(LIVE_S * 5, seed=SEED, duration_s=LIVE_S, fault_rates=BASELINE_FAULTS, start_s=0.0,
                          incidents=[Incident("llm_provider_outage", "llm_error", OUT_LAB[0], OUT_LAB[1], prob=0.7, params={"status": 503, "persistent": True})])
    T_START = time.time(); th = threading.Thread(target=poll_rules, args=(T_START,)); th.start()
    lp.run_live(tk, LIVE_S); print(f"tráfico enviado ({LIVE_S}s); esperando a que la alerta se resuelva…")
    time.sleep(75); STOP.set(); th.join(); lp.close()
    first = {}
    for ts, st in STATES:
        for name, s_ in st.items():
            if s_ in ("firing", "pending") and name not in first: first[name] = round(ts)
    print("primera vez en firing/pending (s desde el inicio):", first, "· el incidente empezó en el s", OUT_LAB[0])
else:
    print("Modo degradado: levanta el stack con:", STACK.command)

In [ ]:
if LIVE:
    from supportops.otel_metrics import prom_range
    fig, axes = plt.subplots(2, 1, figsize=(11, 5.6), sharex=True)
    for name, expr, col_ in (("burn 5 min", burn_promql(DEFAULT_SLOS["availability"], "5m"), viz.PALETTE["traces"]), ("burn 1 min", burn_promql(DEFAULT_SLOS["availability"], "1m"), viz.PALETTE["metrics"])):
        for r in prom_range(expr, minutes=7, step_s=5):
            axes[0].plot([x - T_START for x in r["t"]], r["v"], label=name, color=col_)
    axes[0].axhline(14.4, color=viz.PALETTE["error"], ls=":"); axes[0].axvspan(*OUT_LAB, color=viz.PALETTE["error"], alpha=0.12); axes[0].set_ylabel("burn rate disponibilidad"); axes[0].legend(); axes[0].set_title("Burn rate de disponibilidad (Prometheus) y estado de las alertas (Grafana)")
    names = sorted({n for _, st in STATES for n in st if "availability" in n or "latency" in n})
    for i, n in enumerate(names):
        xs = [ts for ts, st in STATES]; on = [st.get(n) == "firing" for ts, st in STATES]
        axes[1].fill_between(xs, i, i + 0.8, where=on, color=viz.PALETTE["error"], step="mid"); axes[1].text(2, i + 0.35, n, fontsize=7, va="center")
    axes[1].axvspan(*OUT_LAB, color=viz.PALETTE["error"], alpha=0.12); axes[1].set_yticks([]); axes[1].set_xlabel("segundos desde el inicio"); axes[1].set_xlim(0, LIVE_S + 75)
    plt.tight_layout(); plt.show()
    al = backends.grafana_api("/api/alertmanager/grafana/api/v2/alerts")
    print("alertas activas en el Alertmanager de Grafana ahora:", [(a["labels"]["alertname"], a["labels"].get("severity")) for a in al][:6])

> 👀 **Qué debes observar**
>
> - En la gráfica superior el burn de 1 min sube en cuanto empieza el fallo y el de 5 min lo sigue: la alerta de **página** (`fast-burn`) pasa a *firing* cuando **ambos** superan 14,4 y vuelve a *normal* poco después de acabar el incidente, cuando el burn de 1 min cae (la ventana corta apaga la alerta). Los tiempos reales dependen de que Prometheus raspa cada 15 s y Grafana evalúa cada 30 s: detectar en ≈ 1 min en laboratorio equivale a minutos en producción.
> - La alerta de **latencia** no se activa (los errores son rápidos): cada SLO ve lo suyo. La de **disponibilidad lenta** (ticket, ventana de 15 min) dispara más tarde, tras el `for: 1m`, y **tarda más en resolverse**: recuerda 15 min (si repites el experimento antes de que pasen, la verás ya activa al empezar: todavía «recuerda» el incidente anterior).
> - **Notificación:** el punto de contacto del módulo 10 (webhook al endpoint de salud del Collector) recibe las alertas con `severity=page`; en producción sería PagerDuty/Slack. Las alertas activas se ven en el Alertmanager de Grafana (última línea).

### El dashboard `16-slos`

`stack/grafana/dashboards/16-slos.json` (uid `slo-agente`): 4 `stat` con el SLI de cada SLO, 4 `gauge` con el **presupuesto restante**, 4 series de **burn rate** (ventana larga, corta y lenta con umbrales 3 y 14,4), el panel de **alertas** en firing y, de contexto, las trazas con error (Tempo) y los logs de aviso/error (Loki).
**Qué debes ver:** los SLIs en verde en tráfico sano; durante el incidente, los gauges de disponibilidad caen a negativo y la serie de burn cruza la línea roja; las trazas con error enlazan con el span que falló.

---
# 8 · Runbooks, guardia y comunicación

Una alerta sin runbook obliga a pensar a las 3 de la madrugada. Un buen runbook es **corto, ordenado por urgencia y orientado a mitigar** (no a diagnosticar a fondo):

1. **Qué significa** la alerta y qué impacto tiene (en lenguaje del negocio).
2. **Primeros 5 minutos**: tres comprobaciones con enlaces/consultas exactas (dashboard, PromQL, TraceQL).
3. **Árbol de diagnóstico** (síntoma → causa probable → acción).
4. **Mitigación primero** (rollback, conmutación a reserva, degradar a humano), causa raíz después.
5. **Escalado**: a quién, cuándo y con qué criterio de severidad.

In [ ]:
display(Markdown((ROOT / "16_slos_alertas_incidentes" / "runbooks" / "slo-availability-fast-burn.md").read_text(encoding="utf-8")))

## 8.1 La guardia sostenible

| Severidad | Criterio | Respuesta | Canal |
|---|---|---|---|
| **SEV-1** | > 50 % de tickets afectados durante > 15 min, o fuga de datos | Todos los implicados, comunicación a clientes | Página + canal de incidentes |
| **SEV-2** | Quema rápida de un SLO (14,4×) | Guardia primaria, mitigar en < 30 min | Página |
| **SEV-3** | Quema lenta (calidad, coste) | Siguiente día laborable | Ticket |

- **Carga**: el libro *Site Reliability Engineering* (Google, cap. 11 «Being On-Call») orienta a que el equipo no gestione muchas más de un par de incidentes por turno; si tus páginas semanales superan lo que la guardia puede *resolver bien*, reduce ruido antes de añadir gente.
- **Métrica de salud de la guardia**: páginas/semana, **% accionables** (¿hizo falta intervenir?), tiempo de reconocimiento y tiempo de mitigación.
- **Rotación y secundario**: primaria + secundaria; traspaso escrito; tras un SEV-1 la persona de guardia descansa.
- **Comunicación**: un canal de incidentes con un/a responsable de coordinación distinto/a de quien arregla, actualizaciones a horas fijas, y mensaje a clientes en lenguaje llano.

---
# 9 · Simular el incidente y escribir el postmortem

Con los datos del día (la verdad conocida: incidente del proveedor 10:00-10:45) calculamos las cifras del postmortem **desde la telemetría**: cuándo saltó la alerta, cuántos tickets se perdieron y cuánto presupuesto consumió.

In [ ]:
g, t = SERIES["availability"]; sl = DEFAULT_SLOS["availability"]
fire = fire_series(g, t, sl.objective, GOOGLE_POLICIES[0])
o0, o1 = (int(x) for x in TRUTH["outage"])
t_detect = int(np.argmax(fire & (np.arange(NB) >= o0 - 5)))
hh = lambda m: f"{int(m) // 60:02d}:{int(m) % 60:02d}"
bad_inc = int((t[o0:o1] - g[o0:o1]).sum()); tot_inc = int(t[o0:o1].sum())
budget_month = sl.budget * t.sum() * 30
last_fire = int(np.where(fire)[0][-1])
fig, axes = plt.subplots(2, 1, figsize=(11, 5.2), sharex=True)
axes[0].plot(hours, 1 - np.where(t > 0, g / np.maximum(t, 1), np.nan), color=viz.PALETTE["error"], lw=0.8, alpha=0.6); axes[0].set_ylabel("fracción de tickets malos (por minuto)"); axes[0].set_title("Cronología del incidente del proveedor LLM")
axes[1].plot(hours, np.nan_to_num(burn_series(g, t, 60, sl.objective)), label="burn 1 h", color=viz.PALETTE["traces"]); axes[1].plot(hours, np.nan_to_num(burn_series(g, t, 5, sl.objective)), label="burn 5 min", color=viz.PALETTE["metrics"], lw=1)
axes[1].axhline(14.4, color=viz.PALETTE["error"], ls=":"); axes[1].set_ylabel("burn rate"); axes[1].legend(loc="upper right", fontsize=8); axes[1].set_xlabel("hora"); axes[1].set_xlim(9, 13)
for ax in axes:
    ax.axvspan(o0 / 60, o1 / 60, color=viz.PALETTE["error"], alpha=0.1)
    ax.axvline(t_detect / 60, color=viz.PALETTE["ok"], ls="--"); ax.axvline(last_fire / 60, color=viz.PALETTE["neutral"], ls="--")
axes[0].text(t_detect / 60 + 0.02, 0.9, f"alerta {hh(t_detect)}", fontsize=8, color=viz.PALETTE["ok"]); axes[0].text(last_fire / 60 + 0.02, 0.75, f"alerta resuelta {hh(last_fire)}", fontsize=8)
plt.tight_layout(); plt.show()
facts = {"ttd": t_detect - o0, "start": hh(o0), "end": hh(o1), "detect": hh(t_detect), "resolved": hh(last_fire), "bad": bad_inc, "total": tot_inc, "budget_pct": 100 * bad_inc / budget_month}
print(facts)

> 👀 **Qué debes observar**
>
> - **TTD** (tiempo hasta detectar) ≈ 8 min: no es «instantáneo» porque la política exige acumular un 14,4 % de tickets malos en la ventana larga. Si eso es inaceptable para el negocio, la palanca es **bajar el factor o acortar la ventana larga** (a costa de más falsas alarmas) o añadir una sonda sintética que mida cada minuto.
> - Cada minuto de incidente cuesta tickets reales; el postmortem debe traducirlo a **tickets perdidos y a presupuesto consumido** (aquí, ≈ 10 % del presupuesto mensual de disponibilidad en 45 minutos).
> - La alerta se resuelve poco después de que acaba el incidente gracias a la ventana corta: ese es el *tiempo de reajuste*.

In [ ]:
pm = {
    "title": "El proveedor de LLM devuelve 503: el 65 % de los tickets fallan durante 45 min",
    "severity": "SEV-2", "status": "borrador", "authors": "equipo de plataforma de SupportOps",
    "summary": f"Entre las {facts['start']} y las {facts['end']} el proveedor del LLM devolvió errores 503 de forma persistente. {facts['bad']} de {facts['total']} tickets ({facts['bad'] / facts['total']:.0%}) terminaron en error o degradados. "
               f"La alerta de quema rápida de disponibilidad saltó a las {facts['detect']} (TTD {facts['ttd']} min) y se resolvió a las {facts['resolved']}. Se consumió el {facts['budget_pct']:.0f} % del presupuesto mensual de disponibilidad.",
    "impact": [f"{facts['bad']} tickets sin resolver ({facts['bad'] / facts['total']:.0%} del tráfico del periodo); los clientes recibieron «problema técnico» o el mensaje de fallback.",
               f"SLO de disponibilidad: {facts['budget_pct']:.0f} % del presupuesto mensual consumido en 45 minutos.", "Latencia y coste sin impacto (los errores fueron rápidos)."],
    "timeline": [(facts["start"], "Empiezan los 503 del proveedor (primer span `chat` con `error.type=server_error`)."), (hh(o0 + 4), "El burn rate de 5 min supera 14,4×; el de 1 h aún no."),
                 (facts["detect"], f"Salta la alerta `SLO availability · fast-burn` (página). TTD = {facts['ttd']} min."), (hh(t_detect + 3), "La guardia reconoce la alerta y abre el dashboard `slo-agente`."),
                 (hh(t_detect + 8), "Las trazas con error muestran `http.response.status_code=503` en todos los `chat`: causa externa al despliegue."), (facts["end"], "El proveedor se recupera; el burn de 5 min cae por debajo de 14,4×."),
                 (facts["resolved"], "La alerta se resuelve (la ventana corta de 5 min deja de superar el umbral; la de 1 h aún lo supera).")],
    "root_cause": "El proveedor de LLM sufrió una degradación regional. SupportOps tenía un único proveedor y un único modelo, sin conmutación automática a reserva, por lo que el fallo se propagó al 100 % de las llamadas.",
    "contributing": ["No existía un modelo/proveedor de reserva ni un modo degradado automático (escalar a humano).", "La política de reintentos (1 reintento) no ayuda cuando el fallo es persistente y alarga la latencia del error.",
                     "El runbook no indicaba cómo conmutar a la reserva ni quién puede hacerlo."],
    "went_well": ["La alerta multiventana saltó sin falsas alarmas previas y se resolvió sola al terminar el incidente.", "El dashboard enlazado desde la alerta permitió descartar un despliegue en < 10 min."],
    "went_wrong": [f"El TTD de {facts['ttd']} min dejó a clientes sin respuesta; no hubo comunicación proactiva.", "No hay un procedimiento escrito para escalar a humanos en masa."],
    "lucky": ["El incidente terminó antes del máximo de tráfico del mediodía (a las 10:00 ya se está al ≈ 95 % del pico): unas horas más tarde habría afectado a más tickets por minuto."],
    "actions": [("Configurar un proveedor/modelo de reserva con conmutación automática y probarla cada mes", "plataforma", "3 semanas", "mitigar"),
                ("Añadir una sonda sintética por minuto del camino LLM para bajar el TTD de la alerta", "observabilidad", "2 semanas", "detectar"),
                ("Actualizar el runbook de disponibilidad con el procedimiento de conmutación y el árbol de decisión", "guardia", "1 semana", "prevenir"),
                ("Plantilla de comunicación a clientes y a soporte humano para degradaciones de proveedor", "soporte", "2 semanas", "mitigar")]}
display(Markdown(render_postmortem(pm)))

> 👀 **Qué debes observar**
>
> - El postmortem **no nombra a ninguna persona**: describe un sistema con un único proveedor y un runbook incompleto. La pregunta que lo guía es «¿qué hizo fácil que ocurriera y difícil detectarlo?», no «¿quién se equivocó?» (cultura *just culture*; Allspaw, 2012).
> - Las cifras salen de la telemetría (TTD, tickets, presupuesto), no de la memoria de nadie.
> - Cada acción tiene **responsable, plazo y tipo** (prevenir / detectar / mitigar) para poder comprobar si se hizo. Una acción como «tener más cuidado» no es una acción.

---
# 10 · 🏭 En producción

- **Google (SRE Workbook, cap. 5 «Alerting on SLOs»)** describe la evolución desde el umbral fijo hasta la multiventana y multi-burn-rate que usamos; Grafana Labs publica guías para implementarla con Grafana (reglas de grabación por ventana + alertas), y herramientas de código abierto como Sloth y Pyrra generan las reglas a partir de una definición de SLO.
- **Reglas de grabación (*recording rules*).** En producción no se calcula el burn con `rate()` sobre 3 días en cada evaluación: se precalcula el ratio de error por ventana (5 m, 30 m, 1 h, 6 h…) como regla de grabación y las alertas leen esas series. En el laboratorio calculamos las consultas en la propia regla por simplicidad.
- **Dependencias.** Tu disponibilidad está acotada por la de aquello de lo que dependes (proveedor de LLM, herramientas). Un SLO del 99,9 % con un proveedor del 99,5 % necesita una estrategia de reserva.
- **Sistemas multi-agente (módulo 11).** El SLO de extremo a extremo se descompone por agente; la quema del orquestador suele ser consecuencia de la de un subagente: alerta en el *usuario* (síntoma) y diagnostica por *causa*.
- **Postmortems como producto.** Se publican, se revisan en una reunión regular y sus acciones entran en el *backlog* con prioridad; las plataformas de gestión de incidentes automatizan cronologías, pero la disciplina es cultural.

---
# 11 · 🧠 Secretos de la élite

1. **Alerta sobre síntomas del usuario (SLO), diagnostica por causas.** Una página por «CPU alta» es ruido; una por «el presupuesto de disponibilidad se agota en 2 h» es una decisión de negocio. Las alertas de causa (módulo 10) son tickets o paneles, no páginas.
2. **El burn rate es independiente del objetivo; el umbral absoluto no.** Por eso 14,4 sirve para 99 % y para 99,99 %. Si cambias el objetivo, cambian los umbrales absolutos de error, no la política.
3. **La ventana corta es un interruptor de apagado, no de encendido.** Existe para que la alerta se apague en cuanto el problema cesa (≈ 1/12 de la larga). Sin ella tendrías páginas de incidentes ya resueltos durante horas.
4. **Dimensiona las ventanas por el volumen del SLI.** Con pocos eventos (calidad evaluada por muestreo, tráfico nocturno, SLO de un cliente) el burn de una ventana corta es ruido: exige un `min_events` y alarga la ventana, o usa una sonda sintética que genere tráfico de referencia.
5. **Una regresión de calidad es una quema lenta**: no necesita una página a las 3 h sino un **bloqueo de despliegue** (canario con SPRT, módulo 15) y un ticket. Reserva las páginas para lo que no puede esperar a mañana.
6. **El SLO de calidad mide también a tu juez.** Si el juez tiene un 3 % de falsos positivos, un objetivo de 98 % se incumple solo por él. Pon el objetivo por debajo de la línea base medida y vigila `supportops_eval_errors_total`: un juez roto es un falso verde.
7. **SLI de muestra uniforme, muestra de caza aparte.** El SLO usa una muestra simple (sin sesgo, PromQL trivial); la estratificada (errores, valoraciones negativas) sirve para investigar y construir datasets. Mezclarlas obliga a reponderar en cada alerta.
8. **Mide la guardia, no solo el servicio**: páginas por semana, % accionables, tiempo de reconocimiento. Un SLO bien definido con una guardia quemada es un fracaso de fiabilidad.
9. **Postmortem sin culpa ≠ sin responsabilidad.** La responsabilidad es sobre las **acciones** (con dueño y plazo); la culpa es sobre las personas. Si tu postmortem tiene un nombre propio en la causa raíz, reescríbelo.
10. **Congelar con presupuesto agotado solo funciona si se pactó antes.** La política de presupuesto debe estar firmada por producto *antes* del incidente; negociarla en caliente la vacía de contenido. *Plantilla: el Workbook incluye un ejemplo de política de presupuesto de error (apéndice «Example Error Budget Policy»; título según el índice del libro, no abierto desde el sandbox).*
11. **Cultura de guardia con números: ≥ 50 % ingeniería, ≤ 25 % de guardia y ≤ 2 incidentes por turno.** El libro de SRE fija que al menos la mitad del tiempo del equipo sea ingeniería y, del resto, no más del 25 % guardia; el objetivo es como máximo **dos incidentes por turno de guardia (12 h)** porque cada uno, con diagnóstico, mitigación, postmortem y arreglos, consume del orden de 6 horas. Traduce a tu panel de guardia: páginas por turno y % accionables; si superas el límite durante un par de semanas, el síntoma no es «el equipo es lento», es que el servicio debe frenar funcionalidades y comprar fiabilidad. *Fuente: Google SRE Book, cap. 11 «Being On-Call» y cap. 1; cifras confirmadas por búsqueda web (el sitio no abre desde el sandbox); la paráfrasis «2-3 incidentes» de blogs no es del original.*
12. **Entrena la guardia con incidentes simulados.** Google SRE describe el ejercicio «Wheel of Misfortune»: un escenario de fallo reproducido con un compañero que hace de sistema y otro que responde en voz alta; sirve para enseñar a leer señales antes de la primera guardia real. La mesa de incidentes del módulo 21 es esa práctica con tu propia telemetría: mide el tiempo hasta la hipótesis correcta y repítelo por rotación. *Fuente: Google SRE Book, cap. 28 «Accelerating SREs to On-Call and Beyond» (título y ejercicio conocidos del libro; no abierto desde el sandbox).*

---
# 12 · ⚠️ Errores comunes

- SLO del 100 % (no deja presupuesto: es imposible y paraliza los cambios).
- SLOs que no miden lo que ve el usuario (latencia del Collector, CPU) o sin línea base medida.
- Alertar con umbrales fijos de tasa de error: despiertan por ráfagas o llegan tarde.
- Intervalo de grupo de alertas no múltiplo de 10 s (`15s`): Grafana no arranca.
- Alertas sin `__dashboardUid__`/`__panelId__` ni `runbook_url`: el on-call empieza de cero.
- Páginas para quemas lentas (la calidad) y tickets para quemas rápidas (disponibilidad): severidad invertida.
- `noDataState` mal elegido: un SLO sin datos puede ser «OK» (no hay tráfico) o una caída de la telemetría; decide por regla.
- Postmortems con culpables, sin acciones o con acciones sin dueño ni plazo.
- Confiar en el SLO de calidad sin vigilar al juez.

---
# 13 · Autoevaluación

<details>
<summary>1 · (Cálculo) Un SLO de disponibilidad del 99,9 % a 30 días. ¿Cuánto presupuesto de tiempo de indisponibilidad hay? ¿En cuánto tiempo lo agota un burn rate de 14,4 y cuánto presupuesto consume si dura 1 hora?</summary>

Presupuesto: 0,1 % de 43 200 min ≈ **43,2 min**. A 14,4× lo agota en 720 h/14,4 = **50 h**; en 1 h consume 14,4/720 = **2 %** del presupuesto (≈ 52 s de indisponibilidad equivalente). De ahí el 2 % por hora de la política de quema rápida.

</details>

<details>
<summary>2 · (Cálculo) Objetivo 99 %. En la última hora hubo 12 000 tickets y 960 malos; en los últimos 5 minutos, 1 000 tickets y 180 malos. ¿Dispara la política 14,4× (1 h y 5 min)?</summary>

Ventana larga: 960/12 000 = 8 % → burn 8/1 = **8** < 14,4. Ventana corta: 18 % → burn **18**. Como la larga no llega a 14,4, **no dispara**: el problema reciente es grave pero aún no ha acumulado suficiente en la hora; si continúa, la ventana larga cruzará en pocos minutos.

</details>

<details>
<summary>3 · (Diagnóstico) Tu SLO de calidad se incumple cada lunes por la mañana sin ningún despliegue. ¿Qué sospechas?</summary>

Estacionalidad del tráfico: el lunes cambia la **mezcla** (más reembolsos/escalados tras el fin de semana) o el volumen cae y la muestra de calidad es pequeña y ruidosa (`min_events`). También un cambio del juez o de su cobertura. Comprueba la mezcla de intenciones (PSI, módulo 15), el nº de evaluaciones por ventana y la calibración del juez antes de tocar el agente.

</details>

<details>
<summary>4 · (Diagnóstico) La guardia recibió 14 páginas esta semana y 11 se cerraron sin hacer nada. ¿Qué cambias?</summary>

Pasa las alertas no accionables a ticket o dashboard, sustituye umbrales fijos por burn rate multiventana, añade `min_events`, revisa `noDataState` y la agrupación de la política de notificación, y mide el % de acciones útiles por regla. Una regla con < 50 % de accionables se arregla o se elimina.

</details>

<details>
<summary>5 · (Transferencia) Defines un SLO de coste («el 98 % de los tickets cuesta ≤ X»). ¿Qué política de presupuesto asocias y qué alerta?</summary>

Una quema rápida (bucle de herramientas o cambio a un modelo caro) es de página en horario laboral o de ticket urgente según el riesgo financiero; una lenta (deriva de tokens por ticket) es ticket. La acción de mitigación son los *guards* del módulo 13 (tope por ticket, degradación de modelo). Y el SLO se revisa trimestralmente con finanzas.

</details>

<details>
<summary>6 · (Transferencia) ¿Cómo adaptas la política multiventana a un SLO por cliente enterprise con 20 tickets al día?</summary>

Con 20 eventos al día las ventanas de 5 min y 1 h son ruido. Usa ventanas de horas/días, un `min_events` mayor, o un SLI basado en **sondas sintéticas** que generan tráfico de referencia. Alerta además por eventos absolutos (p. ej. ≥ 3 fallos consecutivos) en lugar de ratios.

</details>

<details>
<summary>7 · ¿Qué diferencia hay entre un SLO, un SLA y un objetivo interno de equipo?</summary>

El **SLA** es un contrato con clientes (con consecuencias económicas) y suele ser más laxo; el **SLO** es el objetivo interno más estricto que se vigila para no incumplir el SLA; el SLI es la medida. Mantener el SLO por encima del SLA da margen para reaccionar antes de pagar penalizaciones.

</details>

---
# 14 · Referencias

- Beyer, B., Murphy, N. R., Rensin, D. K., Kawahara, K., Thorne, S. (2018). *The Site Reliability Workbook*, cap. 5 «Alerting on SLOs» — https://sre.google/workbook/alerting-on-slos/ (políticas 14,4×/1 h, 6×/6 h, 1×/3 d).
- Beyer, B., Jones, C., Petoff, J., Murphy, N. R. (2016). *Site Reliability Engineering*, caps. 4 «Service Level Objectives», 11 «Being On-Call» y 15 «Postmortem Culture: Learning from Failure» — https://sre.google/sre-book/table-of-contents/
- Grafana Labs, *How to implement multi-window, multi-burn-rate alerts with Grafana Cloud* (blog, 2025) — https://grafana.com/blog/2025/02/28/how-to-implement-multi-window-multi-burn-rate-alerts-with-grafana-cloud/
- Grafana, documentación de *Alerting → Provision alerting resources* (archivos YAML, `interval`, annotations `__dashboardUid__`/`__panelId__`) — https://grafana.com/docs/grafana/latest/alerting/set-up/provision-alerting-resources/
- Allspaw, J. (2012). *Blameless PostMortems and a Just Culture.* Etsy Code as Craft — https://www.etsy.com/codeascraft/blameless-postmortems/
- OpenTelemetry, *Semantic conventions for generative AI* — métricas `gen_ai.invoke_agent.duration` y `gen_ai.client.inference.duration` usadas como SLIs (estado *Development*).
- Hidalgo, A. (2020). *Implementing Service Level Objectives.* O'Reilly — SLOs, presupuestos y políticas en la práctica.